# A gradient echo on a spiral, assembled here

`sc.modules.SpiralReadout` is **one arm**. A scan is that arm, an excitation in front of it, and a
list of angles — and this notebook writes those three lines rather than importing a class that
hides them. There is deliberately **no `GRESpiral2D`**: strip the angle list out and what remains
is a spoiled gradient echo whose readout happens to be a spiral.

The point of the notebook is the join. `SpiralReadout` says **where the trajectory crosses the
origin**; it does not and must not say where the *echo* is. Aligning the physical echo with the
intended crossing is this layer's job, and getting it wrong produces a legal sequence with the
wrong TE — which is exactly the failure the historical spiral work recorded.

**Output:** one `.seq` file, an eight-interleaf spiral GRE.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pypulseq as pp

import seqcraft as sc

opts = pp.Opts(
    max_grad=38, grad_unit='mT/m',
    max_slew=140, slew_unit='T/m/s',
    rf_dead_time=100e-6,
    rf_ringdown_time=30e-6,
    adc_dead_time=10e-6,
    adc_samples_limit=8192,
)

FOV_MM, MATRIX, THICKNESS_MM = 240.0, 64, 5.0
SHOTS, FLIP_DEG, TR_S = 8, 15.0, 30e-3
DUMMIES = 4

SEQ_DIR = Path('seq')
SEQ_DIR.mkdir(exist_ok=True)
raster = sc.Raster(opts.grad_raster_time)

## The two pieces

The readout is designed once. `shots` enters the **path** — the Nyquist condition is
`dθ/dr = 2π·FOV(r)/shots` — so one interleaf of an eight-shot spiral is a different curve from one
interleaf of a four-shot spiral, not the same curve rotated. The rotation comes later and is a
separate thing.

In [ ]:
exc = sc.modules.Excitation(opts=opts, flip_deg=FLIP_DEG, thickness_mm=THICKNESS_MM,
                            duration_s=1e-3)
spiral = sc.modules.SpiralReadout(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, shots=SHOTS,
                                  dwell_s=4e-6, variant='out')
spoil = sc.modules.spoiler(opts, cycles_per_voxel=4.0, voxel_mm=THICKNESS_MM, axis='z')

print(f'excitation      {exc().duration * 1e3:.3f} ms, centre at {exc.time_to_center() * 1e3:.3f} ms')
print(f'spiral arm      {spiral.duration_s * 1e3:.3f} ms, {spiral.num_samples} samples in '
      f'{len(spiral._segments)} ADC event(s)')
print(f'k_max           {spiral.k_max_per_m:.1f} 1/m   dk {spiral.dk_per_m:.3f} 1/m')
print(f'origin crossing sample {spiral.origin_crossing_samples}, at '
      f'{spiral.time_to_echo(0) * 1e6:.1f} us into the readout block')
print(f'prephase needed {spiral.needs_prephase}   rewind needed {spiral.needs_rewind}')

`variant='out'` starts at the centre of k-space, so its single origin crossing is the **first**
sample and no prephaser is required. It ends at `k_max`, so a rewinder is — and the module states
that requirement and realises it by default.

## TE is owned here, not by the readout

This is the line the whole notebook exists for.

```text
SpiralReadout  owns:  where the trajectory crosses the origin
this layer     owns:  whether the physical echo lands on that crossing
```

So TE is measured from the excitation's **effective centre** to the crossing the readout reports.
Neither number is guessed: both modules report their own, and this cell adds them.

In [ ]:
start_s = float(raster.ceil(exc().duration))
te_s = start_s + spiral.time_to_echo(0) - exc.time_to_center()

print(f'readout starts at   {start_s * 1e3:.3f} ms')
print(f'  + crossing offset {spiral.time_to_echo(0) * 1e6:.1f} us')
print(f'  - RF centre       {exc.time_to_center() * 1e3:.3f} ms')
print(f'  = TE              {te_s * 1e3:.3f} ms')

## One repetition, and the angle schedule

Four lines, and the schedule is a list comprehension. `angle_rad` is a per-call argument, so the
same readout instance serves every interleaf — the same precedent `RadialReadout` set.

In [ ]:
def angles(n):
    """Equal increments over 2 pi: the interleaves of one spiral."""
    return [2.0 * np.pi * i / n for i in range(n)]


def phase_deg(n, increment=117.0):
    """RF spoiling, quadratic in the repetition index."""
    return 0.5 * increment * n * (n + 1)


def repetition(angle_rad, *, phase, acquire=True):
    """Excite, play one interleaf, spoil, fill to TR."""
    out = sc.LogicBlock('spiral_tr').add(0.0, exc(phase_deg=phase))
    out.add(start_s, spiral(angle_rad=angle_rad, acquire=acquire, phase_deg=phase))
    tail = start_s + spiral(angle_rad=angle_rad).duration
    out.add(tail, spoil)
    fill = float(raster.ceil(TR_S - tail - spoil.duration))
    if fill > 1e-9:
        out.add(tail + spoil.duration, pp.make_delay(fill))
    return out


scan = sc.LogicBlock('gre_spiral_2d')
for index in range(DUMMIES):
    scan.add(index * TR_S, repetition(0.0, phase=phase_deg(index), acquire=False))
for index, angle in enumerate(angles(SHOTS)):
    n = DUMMIES + index
    scan.add(n * TR_S, repetition(angle, phase=phase_deg(n)))

print(f'{SHOTS} interleaves + {DUMMIES} dummies, {scan.duration:.3f} s')
print(f'TR {repetition(0.0, phase=0.0).duration * 1e3:.3f} ms')

## What the acquisition encodes

Measured off the **compiled** sequence, not off the design. `sc.kspace` shares no code with the
readout module, so this is a check rather than a restatement.

In [ ]:
seq = sc.compile(scan, opts, name='gre_spiral_2d', definitions={
    'FOV': [FOV_MM / 1e3, FOV_MM / 1e3, THICKNESS_MM / 1e3],
    'TE': te_s, 'TR': TR_S,
})
seq.write(str(SEQ_DIR / 'gre_spiral_2d.seq'))

field = sc.kspace(scan, opts)
k = field['k_adc'][:2].reshape(2, SHOTS, spiral.num_samples)
t_adc = field['t_adc']
radius = np.linalg.norm(k, axis=0)

print(f'{len(seq.block_events)} blocks, {seq.duration()[0]:.3f} s -> seq/gre_spiral_2d.seq')
print()
print(f'|k| at the reported crossing   {radius[:, spiral.echo_sample(0)].max():.4f} 1/m '
      f'= {radius[:, spiral.echo_sample(0)].max() / spiral.dk_per_m:.4f} dk')
print(f'k_max reached                  {radius.max():.2f} 1/m (asked for {spiral.k_max_per_m:.2f})')
print(f'k at the end of every arm      {radius[:, -1].max():.2f} 1/m')

The crossing is inside a sampling window and within a small fraction of one Nyquist step of the
origin — measured on the compiled file, per interleaf.

### The alignment check this notebook exists for

The RF centre of each repetition and the origin crossing of its readout must be `TE` apart. Read
both off the compiled sequence and subtract: if the composition placed the readout against the
wrong instant, this is where it shows, and nothing else in the file would look wrong.

In [ ]:
# Both sides read off the compiled sequence: the excitation instants and the sample times come
# from sc.kspace, which shares no code with either module.  Nothing here restates the arithmetic
# that placed the readout.
rf_centres = field['t_excitation'][-SHOTS:]
crossings = t_adc.reshape(SHOTS, spiral.num_samples)[:, spiral.echo_sample(0)]
measured_te = crossings - rf_centres

print(f'{"shot":>5}  {"RF centre / ms":>15}  {"crossing / ms":>14}  {"TE / ms":>9}')
for shot, (centre, cross, te) in enumerate(zip(rf_centres, crossings, measured_te)):
    print(f'{shot:5d}  {centre * 1e3:15.4f}  {cross * 1e3:14.4f}  {te * 1e3:9.4f}')
print()
print(f'TE spread across interleaves   {np.ptp(measured_te) * 1e9:.3f} ns')
print(f'against the reported TE        {abs(measured_te.mean() - te_s) * 1e9:.3f} ns')


## The interleaves are one contract, rotated

In [ ]:
wanted = np.degrees(angles(SHOTS))
measured = np.degrees(np.arctan2(k[1, :, -1], k[0, :, -1]) - np.arctan2(k[1, 0, -1], k[0, 0, -1]))
print(f'angle error across {SHOTS} interleaves   '
      f'{np.abs((measured - wanted + 180) % 360 - 180).max():.3e} deg')
print(f'extent spread across interleaves  {np.ptp(radius.max(axis=1)):.3e} 1/m')
print(f'samples per interleaf             {spiral.num_samples} (identical by construction)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.6))
for shot in range(SHOTS):
    axes[0].plot(k[0, shot], k[1, shot], lw=0.7, color=plt.cm.viridis(shot / SHOTS))
axes[0].plot(0, 0, 'r+', ms=11)
axes[0].set(title=f'{SHOTS} interleaves, one arm rotated', xlabel='$k_x$ / m$^{-1}$',
            ylabel='$k_y$ / m$^{-1}$', aspect='equal')

axes[1].plot((t_adc[:spiral.num_samples] - t_adc[0]) * 1e3, radius[0], lw=1.0)
axes[1].axhline(spiral.k_max_per_m, color='0.6', ls='--', lw=0.8)
axes[1].plot(0.0, radius[0, spiral.echo_sample(0)], 'r+', ms=11)
axes[1].set(title='one interleaf: |k| against time, crossing marked',
            xlabel='time from the first sample / ms', ylabel='$|k|$ / m$^{-1}$')
fig.tight_layout()

## What this notebook established

| | |
|---|---|
| a spiral GRE is **one arm plus a schedule** | which is why no `GRESpiral2D` ships — it would own the angle list and no physics |
| **TE is owned here** | measured from the excitation's effective centre to the crossing the readout reports, with both numbers read off their own modules |
| the echo lands where it was placed | TE identical across interleaves to nanoseconds, and matching the reported value |
| interleaves are one contract, rotated | angle error below 1e-12 degrees, identical extent and sample count |
| the crossing is inside a sampling window | measured on the compiled file, a small fraction of `dk` from the origin |
| the whole acquisition compiles | one `.seq`, legal Pulseq |

There is **no `02_simulate_and_reconstruct.ipynb`**, and that is deliberate. A spiral image needs
a non-Cartesian reconstruction, and the example suite has none to reuse — every existing `02` is
an FFT on a Cartesian grid. Reviewing that path is a separate task, and it now has **two** real
consumers, `RadialReadout` and `SpiralReadout`, which is the condition for looking at whether a
shared contract exists at all. Promotion into the package is not a predetermined outcome;
example-only remains a valid answer.